In [61]:
import sys
from pathlib import Path

In [62]:
src_path = Path("../src").resolve()
sys.path.append(str(src_path))

In [63]:
from sqlmodel import Session, col, select
from sqlalchemy import asc, desc, func
from sqlmodel import create_engine
from timescaledb.hyperfunctions import time_bucket
from pprint import pprint
from datetime import datetime, timedelta, timezone 

from api.db.session import engine
from api.events.models import EventModel

In [64]:
engine = create_engine(engine.url.set(host="localhost"))

with Session(engine) as session:
    query = select(EventModel).order_by(col(EventModel.updated_at).asc()).limit(10)
    compiled_query = query.compile(compile_kwargs={"literal_binds": True})
    print(compiled_query)
    print("")
    print(str(query))
    # results = session.exec(query).all()
    # print(results)

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at ASC
 LIMIT 10

SELECT eventmodel.id, eventmodel.time, eventmodel.page, eventmodel.description, eventmodel.updated_at 
FROM eventmodel ORDER BY eventmodel.updated_at ASC
 LIMIT :param_1


In [65]:
with Session(engine) as session:
    bucket = time_bucket("1 hour", EventModel.time) # Buckets of data are chunked by a time range of 1 hour
    pages = ['/about', '/contact', '/pages', 'pricing'] 
    start = datetime.now(timezone.utc) - timedelta(hours=2) # Filters the time range of data
    finish = datetime.now(timezone.utc) + timedelta(hours=1) # Time bucket does this as well
    query = (
        select(
            bucket, 
            EventModel.page,
            func.count() # .label("event_count")
        )
        .where(
            EventModel.time > start,
            EventModel.time <= finish,
            col(EventModel.page).in_(pages) # Filter by page
        )
        .group_by(
            bucket,
            EventModel.page
        ) # Group
        .order_by(
            bucket,
            EventModel.page
        )
    )
    compiled_query = query.compile(compile_kwargs={"literal_binds": True}) # Compiles query into valid SQL
    print(compiled_query)
    results = session.exec(query).fetchall()
    pprint(results)

SELECT time_bucket(INTERVAL '1 hour', eventmodel.time) AS time_bucket_1, eventmodel.page, count(*) AS count_1 
FROM eventmodel 
WHERE eventmodel.time > '2026-09-30 22:58:42.340134+00:00' AND eventmodel.time <= '2026-10-01 01:58:42.340155+00:00' AND eventmodel.page IN ('/about', '/contact', '/pages', 'pricing') GROUP BY time_bucket(INTERVAL '1 hour', eventmodel.time), eventmodel.page ORDER BY time_bucket(INTERVAL '1 hour', eventmodel.time), eventmodel.page
[]
